# Risk Score Pipeline — `amygda_ops_risk_score`

## What this notebook does

Continues from the labelling notebook.  It takes the **trained labelling model zip**
produced by `run_classification()`, trains a risk score model on your historical data,
and generates a `risk_scores.parquet` file with per-asset, per-period operational
risk scores across every system and subsystem in the hierarchy.

## What you need before starting

- Your permanent `api_key` (same one used in the labelling notebook)
- The `trained_labelling_model_ses-xxx.zip` downloaded by `run_classification()`

Both are loaded automatically from the labelling artifact directory — no copy-pasting needed.

## Step overview

| Step | Method | Re-runnable? | Locked after |
|------|--------|--------------|-------------|
| 9  | `configure_training` | Yes | `train_risk_model` starts |
| 10 | `train_risk_model` | Yes | `configure_generation` starts |
| 11 | `configure_generation` | Yes | `generate_risk_scores` starts |
| 12 | `generate_risk_scores(dest_dir)` | **ONE-WAY DOOR** | — |

**Two zip types are accepted by `import_model()`:**

| Zip type | Unlocks |
|----------|---------|
| `trained_labelling_model_ses-xxx.zip` | Steps 9–12 (full training required) |
| `complete_model_ses-xxx.zip` | Step 11 only (training already done, skip to generation) |

## Ongoing artifacts

```
artifacts/risk_score/
  import_model.json
  configure_training.json
  train_risk_model.json
  calibration_thresholds.json   ← auto-downloaded after training
  configure_generation.json
  generate_risk_scores.json
  risk_scores.parquet            ← auto-extracted from zip
  classified_logs.parquet        ← free-text mode only
  logs_by_system_subsystem.json
  accepted_hierarchy.json        ← hierarchy used for scoring
```

## Session lifecycle guide

Same rules as the labelling notebook — read this once.

---

### Every time you open this notebook — run Cell 1 first (imports)

---

### API readiness — always call `client.wait_until_ready()` first

The API uses scale-to-zero.  On a cold start ML models take 60–120 s to load.
`wait_until_ready()` polls until ready and returns immediately on a warm instance.

---

### Case 1 — Starting a new risk score session

Use **Cell 2** (the setup cells below).  The `api_key` is the same permanent key
from the labelling notebook — no re-registration needed.

---

### Case 2 — Kernel restarted

Use **Cell 3** (kernel-restart cell).  Paste your `AUTH_ID` and run.
If it raises `SessionNotFoundError`, the session TTL has expired — use Cell 2
to open a fresh session and call `import_model(ZIP_PATH)` again.

---

### Case 3 — Network dropped and came back

Your in-memory `session` object is intact — just continue.
If `session.status()` raises `APIError (404)`, call `client.open_session()` again;
it auto-resumes if still alive or creates a fresh one.  **No re-registration needed.**

---

### Case 4 — Step is locked, need to go back

```python
session = client.restart_session(session, api_key=API_KEY, config=config)
# All locks cleared — re-run from import_model().
```
⚠ Permanently deletes server-side session data.  `api_key` is never affected.

---

### Note — `next_step` in every response

Every step method returns a dict that includes a `next_step` key telling you what to call next:

```python
result = session.configure_training(...)
print(result['next_step'])  # e.g. 'run_training'
```

| Step completed | `next_step` value |
|---|---|
| `import_model` | `configure_training` |
| `configure_training` | `train_risk_model` |
| `train_risk_model` | `configure_generation` |
| `configure_generation` | `generate_risk_scores` |
| `generate_risk_scores` | `done` |

> **Note:** `next_step` values match both the session status step key and the SDK method name exactly..

In [ ]:
# ── Cell 1: Install and import ────────────────────────────────────────────────
# Run once at the start of every session.
# --upgrade ensures you always have the latest version of the SDK.
#!pip install --upgrade git+https://github.com/amygda/amygda_ops_risk_score_sdk.git

import json, os
from amygda_ops_risk_score import OpsRiskClient, SessionConfig
from amygda_ops_risk_score import helpers

print("SDK ready.")

In [ ]:
helpers.plot_asset_risk

In [ ]:
# ── Cell 2a: Connect and wait for the API ────────────────────────────────────
client = OpsRiskClient()
client.wait_until_ready()
print("API is ready.")

In [ ]:
# ── Cell 2b: Load your credentials from the labelling artifacts ──────────────
# api_key is the same permanent key from 01_labelling.ipynb.
#
# INTERNAL TESTING — two keys are already live in the hosted API:
#   yk-amygda-dev-normal   ($20 balance — use this for normal testing)
#   yk-amygda-dev-low      ($4 balance  — triggers InsufficientCreditsError immediately)
# PRODUCTION: get your key from the Amygda portal (https://portal.amygda.io — coming soon).
# api_key is PERMANENT — it survives session restarts and never needs to be recreated.

ARTIFACT_DIR_LABELLING = "my_artifacts/labelling/"   # ← same path used in 01_labelling

with open(f"{ARTIFACT_DIR_LABELLING}api_key.txt") as f:
    API_KEY = f.read().strip()
# Or paste directly:
# API_KEY = "yk-amygda-dev-normal"

with open(f"{ARTIFACT_DIR_LABELLING}run_classification.json") as f:
    ZIP_PATH = json.load(f)["zip_path"]
# Or paste directly:
# ZIP_PATH = "outputs/trained_labelling_model_ses-xxxx.zip"  ← labelling zip
# ZIP_PATH = "outputs/complete_model_ses-xxxx.zip"           ← complete model zip (skips training)

ARTIFACT_DIR = "my_artifacts/risk_score/"   # ← folder for all step JSON artifacts

print(f"API_KEY  : {API_KEY}")
print(f"ZIP_PATH : {ZIP_PATH}")
print(f"ARTIFACT_DIR: {ARTIFACT_DIR}")

In [ ]:
# ── Cell 2c: Open a session and import the model zip ────────────────────────
# open_session() creates a new session or auto-resumes the existing one.

config  = SessionConfig(name="risk-score-run")   # ← give it a meaningful name
session = client.open_session(api_key=API_KEY, config=config, artifact_dir=ARTIFACT_DIR)

AUTH_ID = session.auth_id
print(f"AUTH_ID  : {AUTH_ID}")
print("Save AUTH_ID — needed if your kernel restarts.")
print()

# Import the trained model zip.  This is a file upload only — no recomputation.
# import_result["next_step"] tells you where to continue:
#   "configure_training"   → labelling zip; run Steps 9-12
#   "configure_generation" → complete model zip; training already done, skip to Step 11
import_result = session.import_model(ZIP_PATH)
# import_result is automatically saved to {ARTIFACT_DIR}import_model.json
print(json.dumps(import_result, indent=2))
print(f"Next step: {import_result.get('next_step')}")

In [ ]:
# ── Cell 3: Kernel restarted — reconnect to your existing session ─────────────
# Use this cell (instead of Cell 2) when your kernel restarted but the session
# on the server is still alive.
#
# 1. Paste the AUTH_ID printed when you ran Cell 2c.
# 2. Set paths to match Cell 2b.
# 3. Run this cell.
# ─────────────────────────────────────────────────────────────────────────────

AUTH_ID              = "ses-paste-your-auth-id"   # ← paste your saved AUTH_ID
API_KEY              = "paste-your-api-key-here"   # ← paste your saved API_KEY
ARTIFACT_DIR         = "my_artifacts/risk_score/"     # ← same path as Cell 2b
ARTIFACT_DIR_LABELLING = "my_artifacts/labelling/"    # ← same path as Cell 2b

with open(f"{ARTIFACT_DIR_LABELLING}run_classification.json") as f:
    ZIP_PATH = json.load(f)["zip_path"]

client  = OpsRiskClient()
client.wait_until_ready()

session = client.get_session(AUTH_ID, artifact_dir=ARTIFACT_DIR)

status = session.status()
print(json.dumps(status, indent=2))
print()
print("Step state guide:")
print("  DONE    → completed — skip this step")
print("  RUNNING → in-flight on server — call session.status() again in a moment")
print("  FAILED  → failed — safe to re-run")
print("  NONE    → not started — proceed normally")
print()
print("If get_session() raised SessionNotFoundError, the TTL expired.")
print("Run Cell 2a → 2b → 2c to open a fresh session and call import_model(ZIP_PATH) again.")

## Step 9 — configure_training

Upload your historical log data and configure the training parameters.  The training
step uses this data to learn per-subsystem calibration thresholds — the boundaries
that separate normal from elevated risk for each system component.

**Key parameters:**
- `asset_id_column` — unique identifier for each asset (e.g. train number, vehicle ID)
- `timestamp_column` — when the log entry occurred; used to build rolling time windows
- `rolling_window` — days to aggregate events over (1–30, default 7)
- `rolling_feature_type` — how events are aggregated:
  - `"sum"` — total event count in the window
  - `"flag"` — 1 if any event occurred, 0 otherwise
  - `"ewm"` — exponentially weighted mean; more weight on recent days *(fixed-log only)*
  - `"all"` — runs sum + flag + ewm together *(fixed-log only)*
  - **Free-text sessions only support `"sum"` and `"flag"`** — `"ewm"`/`"all"` will be rejected.
- `quantile_for_thresholds` — calibration percentile (0.5–1.0, default 0.99 = top 1%)

**Validation rules:**
- `asset_id_column` and `timestamp_column` must exist in the file headers
- `rolling_window` must be an integer between 1 and 30
- `quantile_for_thresholds` must be a float in [0.5, 1.0)

**Re-run rule:** Can re-run anytime until `train_risk_model` starts.

In [ ]:
# ── Step 9 params — edit here ─────────────────────────────────────────────────
TRAINING_FILE           = "../sample_data/risk_score_training.csv"  # historical logs (.csv / .xlsx)
ASSET_ID_COLUMN         = "asset_id"           # unique asset identifier column
TIMESTAMP_COLUMN        = "timestamp"          # event timestamp column
DATE_FORMAT             = "%d/%m/%Y"                # "infer" auto-detects; or provide strftime format
ROLLING_WINDOW          = 7                    # days to aggregate events over (1–30)
ROLLING_FEATURE_TYPE    = "ewm"                # "sum" | "flag" | "ewm" | "all"  (ewm/all: fixed-log sessions only)
QUANTILE_FOR_THRESHOLDS = 0.99                 # calibration percentile: 0.99 = top 1 %
TRAINING_SHEET          = None                 # XLSX only — None = single sheet

training_config_result = session.configure_training(
    file_path=TRAINING_FILE,
    asset_id_column=ASSET_ID_COLUMN,
    timestamp_column=TIMESTAMP_COLUMN,
    date_format=DATE_FORMAT,
    rolling_window=ROLLING_WINDOW,
    rolling_feature_type=ROLLING_FEATURE_TYPE,
    quantile_for_thresholds=QUANTILE_FOR_THRESHOLDS,
    sheet_name=TRAINING_SHEET,
)
# training_config_result is automatically saved to {ARTIFACT_DIR}configure_training.json
print(json.dumps(training_config_result, indent=2))

## Step 10 — Risk Score Model Training  _(background)_

Train the risk score model.  The server classifies every row in the training dataset
against the hierarchy, builds rolling time-window features per asset, and calibrates
per-subsystem thresholds at the quantile you configured.

The SDK polls until training completes and prints rotating progress messages.
Typical runtime: 1–10 minutes depending on dataset size.

The result includes:
- `subsystems_calibrated` — how many subsystems received calibration thresholds
- `thresholds_path` — local path to `calibration_thresholds.json` in `artifact_dir`

**Re-run rule:** Can re-run anytime until Step 11 starts.  
**Locked once:** `configure_generation` is called.

**Kernel restart recovery:** If your kernel restarts while training is running,
check `session.status()` first.  If training is `DONE`, load from the saved artifact:

```python
with open(f'{ARTIFACT_DIR}train_risk_model.json') as f:
    training_result = json.load(f)
```

In [ ]:
training_result = session.train_risk_model(timeout=3600)  # Cloud Run hard limit is 3600 s
# training_result is automatically saved to {ARTIFACT_DIR}train_risk_model.json
print(json.dumps(training_result, indent=2))


In [ ]:
# ── Inspect calibration thresholds ────────────────────────────────────────────
# Mirrors the "View Thresholds" panel on the Streamlit Risk Score page.
# calibration_thresholds.json is downloaded automatically after train_risk_model().

import os

thresholds_path = training_result.get("thresholds_path") or f"{ARTIFACT_DIR}calibration_thresholds.json"

if os.path.exists(thresholds_path):
    with open(thresholds_path) as f:
        thresholds = json.load(f)
    print(f"Subsystems calibrated: {len(thresholds)}")
    print()
    # Show as sorted table
    import pandas as pd
    df_thresh = (
        pd.DataFrame(list(thresholds.items()), columns=["subsystem_pair", "threshold"])
        .sort_values("threshold", ascending=False)
        .reset_index(drop=True)
    )
    print(df_thresh.to_string(index=False))
else:
    print(f"Thresholds file not found at {thresholds_path}")
    print("Re-run train_risk_model() or check ARTIFACT_DIR.")

## Training Visualisations

After training completes, use these helpers to inspect the calibrated thresholds and the
feature-engineered training dataset.  All helpers accept file paths — they work after a
kernel restart without re-running `train_risk_model()`.

```python
thresholds_path  = training_result.get("thresholds_path")   or f"{ARTIFACT_DIR}calibration_thresholds.json"
training_fe_path = training_result.get("training_fe_path")  or f"{ARTIFACT_DIR}training_fe.parquet"
```

In [ ]:
# ── 1. Threshold summary — thresholds vs mean/max per subsystem ─────────────────────
# Edit systems= to match the system names in your hierarchy.
thresholds_path  = training_result.get("thresholds_path")  or f"{ARTIFACT_DIR}calibration_thresholds.json"
training_fe_path = training_result.get("training_fe_path") or f"{ARTIFACT_DIR}training_fe.parquet"

helpers.plot_threshold_summary(
    thresholds_path,
    training_fe_path,
    systems=None,    # ← None = all systems; or pass a list e.g. ['motion_control','baggage_handling']
)

In [ ]:
# ── 2. Feature distributions — where does the threshold sit in each subsystem? ──
helpers.plot_feature_distributions(
    training_fe_path,
    thresholds_path,
    systems=None,    # ← None = all systems; or filter to specific ones
)

In [ ]:
# ── 3. Quick data inspection ──────────────────────────────────────────────────
import pandas as pd

training_fe = pd.read_parquet(training_fe_path)

print("Risk feature columns (first 10):")
print([x for x in training_fe.columns if "risk_feature" in x][:10])
print()
print("Unique assets in training data:")
print(training_fe["asset_id"].unique())

In [ ]:
# ── 4. Feature time-series — rolling feature values over time for an asset ────
# Edit asset_id, system, and subsystem to match your data.

INSPECT_ASSET   = ["ASSET-001", "ASSET-002", "ASSET-003"],         # ← edit: one or more asset IDs
INSPECT_SYSTEM  = "control_and_recovery"          # ← edit: system name from hierarchy
INSPECT_SUBSYS  = "fault_handling"       # ← edit: subsystem name


helpers.plot_feature_timeseries(
    training_fe_path,
    thresholds_path,
    asset_id=["ASSET-001", "ASSET-002"],
    system=INSPECT_SYSTEM,
    subsystem=INSPECT_SUBSYS,
)

In [ ]:
# ── 5. Feature breakdown — per-asset feature detail for one subsystem ───────────
helpers.plot_feature_breakdown(
    training_fe_path,
    thresholds_path,
    asset_id="ASSET-001",
    system=INSPECT_SYSTEM,
    subsystem=INSPECT_SUBSYS,
    is_free_text=training_config_result["is_free_text"],
    logs_mapping=training_result.get("logs_mapping_path"),
)

## Step 11 — configure_generation

Upload the logs you want to score — this can be the same dataset used for training
or a completely new batch of operational logs.  The trained thresholds from Step 10
are applied to this data to produce risk scores.

The log file must have the same structure as the training file (same columns) because
it goes through the same classification pipeline.

**Re-run rule:** Can re-run anytime until Step 12 starts.  
**Locked once:** `run_generation` is called.

In [ ]:
# ── Step 11 params — edit here ────────────────────────────────────────────────
GENERATION_FILE  = "../sample_data/risk_score_prediction.csv"   # logs to score (.csv / .xlsx)
DATE_FORMAT      = "%d/%m/%Y" 
GENERATION_SHEET = None   # XLSX only

generation_config_result = session.configure_generation(
    file_path=GENERATION_FILE,
    date_format=DATE_FORMAT,
    sheet_name=GENERATION_SHEET,
)
# generation_config_result is automatically saved to {ARTIFACT_DIR}configure_generation.json
print(json.dumps(generation_config_result, indent=2))

## Step 12 — Risk Score Generation  🔒 ONE-WAY DOOR

**Cannot be repeated on the same session.**

Generate risk scores for every asset and time period in the generation dataset.

The SDK will:
1. Trigger generation on the server (runs in background)
2. Poll until complete — prints rotating progress messages
3. **Auto-download** `complete_model_{auth_id}.zip` into `dest_dir`
4. **Auto-extract** into `artifact_dir` (if set):
   - `risk_scores.parquet` → `result["parquet_path"]`
   - `classified_logs.parquet` → `result["classified_logs_path"]` (free-text only)
   - `logs_by_system_subsystem.json` → `result["logs_mapping_path"]`
   - `calibration_thresholds.json` → `result["thresholds_path"]`
   - `accepted_hierarchy.json` → `result["hierarchy_path"]`
5. **Auto-wipe** the session

After any future kernel restart, plot directly from the artifact:

```python
helpers.plot_asset_risk_ranking(f"{ARTIFACT_DIR}risk_scores.parquet")
```

**To re-run generation on new data** (same thresholds, no re-training):
see the **Re-scoring** section at the bottom of this notebook.

In [ ]:
OUTPUT_DIR = "my_outputs/"    # directory to save the complete model zip into

generation_result = session.generate_risk_scores(OUTPUT_DIR, timeout=3600)
# generation_result is automatically saved to {ARTIFACT_DIR}generate_risk_scores.json

zip_path     = generation_result["zip_path"]
parquet_path = generation_result.get("parquet_path")

print(f"Complete model zip:    {zip_path}")
print(f"Risk scores parquet:   {parquet_path}")
print(f"Thresholds:            {generation_result.get('thresholds_path')}")
print(f"Hierarchy:             {generation_result.get('hierarchy_path')}")
print(f"Logs mapping:          {generation_result.get('logs_mapping_path')}")
print("Session wiped automatically.")

In [ ]:
helpers.plot_asset

## Visualise risk scores

All plot helpers accept a file path or a DataFrame directly.  The file path form works
even after a kernel restart — no need to re-run `run_generation()`.

```python
# After kernel restart — no re-run needed:
helpers.plot_asset_risk_ranking(f"{ARTIFACT_DIR}risk_scores.parquet")

# If you have the DataFrame already loaded:
import pandas as pd
risk_df = pd.read_parquet(f"{ARTIFACT_DIR}risk_scores.parquet")
helpers.plot_asset_risk_ranking(risk_df)
```

In [ ]:
# ── Plot from the parquet file (also works after kernel restart) ──────────────
parquet_path = generation_result.get("parquet_path") or f"{ARTIFACT_DIR}risk_scores.parquet"

# Bar chart: top-20 assets by mean operational_risk
helpers.plot_asset_risk_ranking(
    parquet_path,
    aggregation="max",
    top_n=20,
    title="Asset Operational Risk Scores",
)

# Histogram: distribution of risk scores across all records
helpers.plot_risk_score_distribution(
    parquet_path,
    title="Risk Score Distribution",
)

In [ ]:
# ── Load parquet for custom analysis ─────────────────────────────────────────
import pandas as pd

parquet_path = generation_result.get("parquet_path") or f"{ARTIFACT_DIR}risk_scores.parquet"
risk_df = pd.read_parquet(parquet_path)

print(f"Risk scores: {len(risk_df):,} rows, {risk_df['asset_id'].nunique()} assets")
print(f"Columns: {list(risk_df.columns[:10])} ... ({len(risk_df.columns)} total)")
risk_df.head()

## Visual Analysis

All five helpers below accept a file path **or** a DataFrame — file paths work after a kernel restart without re-running `run_generation()`.

```python
parquet_path         = generation_result.get("parquet_path")          or f"{ARTIFACT_DIR}risk_scores.parquet"
classified_logs_path = generation_result.get("classified_logs_path")  or f"{ARTIFACT_DIR}classified_logs.parquet"
```

### Helpers overview

| Helper | Description | Mode |
|--------|-------------|------|
| `plot_risk_heatmap` | Heatmap: all assets × all dates | Both |
| `plot_daily_risk_snapshot` | Tile panel for one specific date | Both |
| `plot_asset_risk_over_time` | Heatmap: all risk types × all dates for one asset | Both |
| `plot_subsystem_log_activity` | Stacked bar of log code counts per subsystem | Fixed-log only |
| `get_logs_for_subsystem` | Raw log text table for a subsystem + date window | Free-text only |

In [ ]:
# ── Multi-Asset Risk Heatmap Over Time ───────────────────────────────────────
# Rows = assets, columns = dates, colour = chosen risk metric.
#
# metric= accepts:
#   "operational_risk"           — overall aggregate score (default)
#   "{system}_system_risk"       — e.g. "conveyor_handling_system_risk"
#                                   run helpers.list_risk_metrics(parquet_path) to see all
#
parquet_path = generation_result.get("parquet_path") or f"{ARTIFACT_DIR}risk_scores.parquet"

# See every valid metric value for this parquet:
print("Available metrics:")
for m in helpers.list_risk_metrics(parquet_path):
    print(f"  {m}")
print()

helpers.plot_risk_heatmap(
    parquet_path,
    metric="operational_risk",          # ← swap to any {system}_system_risk column
    # asset_ids=["K204", "K205"],       # optional: limit to specific assets
    title="Multi-Asset Operational Risk Heatmap",
)

In [ ]:
# ── Risk Scores by Date ───────────────────────────────────────────────────────
# Tile panel for a single date: one large op-risk tile per asset,
# with system-level tiles below each asset.
# Colour: blue (0-33), yellow (33-66), red (66-100).

SCORE_DATE = "2025-01-09"          # ← edit: the date you want to inspect
MIN_SCORE  = 0.0                   # ← hide assets below this operational_risk threshold

helpers.plot_daily_risk_snapshot(
    parquet_path,
    date=SCORE_DATE,
    min_score=MIN_SCORE,
    # asset_ids=["asset-001"],     # optional filter
    title="Risk Scores by Date",
)

In [ ]:
# ── Single Asset Risk Analysis ────────────────────────────────────────────────
# Heatmap for one asset: rows = risk types (Operational Risk at top,
# then each system), columns = dates.

ASSET_TO_INSPECT = "K236"    # ← edit: the asset you want to drill into

helpers.plot_asset_risk_over_time(
    parquet_path,
    asset_id=ASSET_TO_INSPECT,
    title=f"Risk Analysis — {ASSET_TO_INSPECT}",
)

helpers.plot_asset_risk_over_time(
    parquet_path,
    asset_id=ASSET_TO_INSPECT,
    weighted=True,
    model_config=f"{ARTIFACT_DIR}model_config.json",
    title=f"Risk Analysis — Weighted {ASSET_TO_INSPECT}",
)

In [ ]:
# ── System-level heatmap for one asset ───────────────────────────────────────────
# Rows = subsystem risk scores, columns = dates.
# Edit SYSTEM to any system name from the hierarchy (snake_case).

SYSTEM = "baggage_handling"   # ← edit: system name from hierarchy

helpers.plot_asset_system_over_time(
    parquet_path,
    asset_id=ASSET_TO_INSPECT,
    system=SYSTEM,
    title=f"{SYSTEM.replace('_',' ').title()} — {ASSET_TO_INSPECT}",
)

helpers.plot_asset_system_over_time(
    parquet_path,
    asset_id=ASSET_TO_INSPECT,
    system=SYSTEM,
    weighted=True,
    model_config=f"{ARTIFACT_DIR}model_config.json",
    title=f"{SYSTEM.replace('_',' ').title()} — Weighted {ASSET_TO_INSPECT}",
)

In [ ]:
# ── Load model_config.json for inspection ─────────────────────────────────────────
import json as _json

model_config_path = generation_result.get("model_config_path") or f"{ARTIFACT_DIR}model_config.json"

with open(model_config_path) as _f:
    model_config = _json.load(_f)

print("model_config.json loaded.")

In [ ]:
# ── Operational risk breakdown — contribution of each system for one asset ───
helpers.plot_system_contributions(
    parquet_path,
    asset_id=ASSET_TO_INSPECT,
    model_config=model_config_path,
)

In [ ]:
# ── Risk Breakdown — Asset + Date ────────────────────────────────────────────
# Full operational risk + every system and subsystem score for one asset on one date.
# No need to know any system or subsystem names in advance.
# If the exact date is not in the parquet, the closest available date is used.

BREAKDOWN_ASSET = "K397"        # ← edit: asset to inspect
BREAKDOWN_DATE  = "2025-09-19"  # ← edit: date to inspect ("YYYY-MM-DD")

breakdown = helpers.get_risk_breakdown(parquet_path, BREAKDOWN_ASSET, BREAKDOWN_DATE)

import json
print(json.dumps(breakdown, indent=2))

# ── Flat DataFrame view ───────────────────────────────────────────────────────
import pandas as pd

rows = [{"level": "operational", "name": "operational_risk",
         "score": breakdown["operational_risk"]}]
for sys_name, score in breakdown["systems"].items():
    rows.append({"level": "system", "name": sys_name, "score": score})
for pair, score in breakdown["subsystems"].items():
    rows.append({"level": "subsystem", "name": pair, "score": score})

df_breakdown = (
    pd.DataFrame(rows)
    .sort_values(["level", "score"], ascending=[True, False])
    .reset_index(drop=True)
)
#display(df_breakdown)

In [ ]:
# ── Log Occurrences — Subsystem Breakdown ────────────────────────────────────
# Auto-detects mode from the source file:
#   Free-text mode  → pass classified_logs.parquet
#                     shows daily count of log entries for the subsystem
#   Fixed-log mode  → pass risk_scores.parquet + logs_mapping
#                     shows stacked bar of individual log-code daily counts
import os
parquet_path         = generation_result.get("parquet_path")         or f"{ARTIFACT_DIR}risk_scores.parquet"
classified_logs_path = generation_result.get("classified_logs_path") or f"{ARTIFACT_DIR}classified_logs.parquet"
logs_mapping_path    = generation_result.get("logs_mapping_path")    or f"{ARTIFACT_DIR}logs_by_system_subsystem.json"

ASSET_ID   = "K397"                      # ← edit
SYSTEM     = "motion_control"            # ← edit: system name from the hierarchy
SUBSYSTEM  = "datum_homing"             # ← edit: subsystem name
END_DATE   = "2025-09-19"               # ← edit: end of the window
DAYS_BACK  = 60                          # rolling window in days
LOG_COLUMN = "cleaned_event_details"     # ← free-text mode only; ignored in fixed-log mode

if os.path.exists(classified_logs_path):
    # Free-text mode: classified_logs.parquet exists
    # helpers.plot_subsystem_log_activity(
    #     classified_logs_path,
    #     asset_id=ASSET_ID, system=SYSTEM, subsystem=SUBSYSTEM,
    #     date=END_DATE, days_back=DAYS_BACK, log_column=LOG_COLUMN,
    #     title=f"Log Occurrences — {ASSET_ID} / {SYSTEM} / {SUBSYSTEM}",
    # )
    helpers.plot_subsystem_log_activity(
    classified_logs_path,
    asset_id=ASSET_ID, system=SYSTEM, subsystem=SUBSYSTEM,
    date=END_DATE, days_back=DAYS_BACK, log_column=LOG_COLUMN,
    risk_source=parquet_path,   # <-- add this
    title=f"Log Occurrences — {ASSET_ID} / {SYSTEM} / {SUBSYSTEM}",
)
elif os.path.exists(parquet_path) and os.path.exists(logs_mapping_path):
    # Fixed-log mode: use risk_scores.parquet + logs_by_system_subsystem.json
    # helpers.plot_subsystem_log_activity(
    #     parquet_path,
    #     asset_id=ASSET_ID, system=SYSTEM, subsystem=SUBSYSTEM,
    #     date=END_DATE, days_back=DAYS_BACK,
    #     logs_mapping=logs_mapping_path,
    #     title=f"Log Occurrences — {ASSET_ID} / {SYSTEM} / {SUBSYSTEM}",
    # )
    helpers.plot_subsystem_log_activity(
    parquet_path,
    asset_id=ASSET_ID, system=SYSTEM, subsystem=SUBSYSTEM,
    date=END_DATE, days_back=DAYS_BACK, log_column=LOG_COLUMN,
    risk_source=parquet_path,   # <-- add this
    logs_mapping=logs_mapping_path,
    title=f"Log Occurrences — {ASSET_ID} / {SYSTEM} / {SUBSYSTEM}",)
else:
    print("Source files not found. Check ARTIFACT_DIR and re-run generation.")

In [ ]:
# ── Raw Log Table (free-text mode only) ──────────────────────────────────────
# Returns a DataFrame of classified raw log entries for a chosen asset /
# system / subsystem within a date window.
# classified_logs.parquet is extracted to artifact_dir by run_generation()
# only when is_free_text=True was used in the labelling notebook.

import os

classified_logs_path = generation_result.get("classified_logs_path") or f"{ARTIFACT_DIR}classified_logs.parquet"

ASSET_ID   = "K236"                      # ← edit
SYSTEM     = "motion_control"            # ← edit: system name from the hierarchy
SUBSYSTEM  = "axis_control"             # ← edit: subsystem name
END_DATE   = "2025-12-09"               # ← edit: end of the window
DAYS_BACK  = 90                          # rolling window in days
LOG_COLUMN = "cleaned_event_details"     # ← match the log_column used in configure()

if not os.path.exists(classified_logs_path):
    print("classified_logs.parquet not available — this helper is for free-text mode only.")
    print(f"Expected at: {classified_logs_path}")
else:
    logs_df = helpers.get_logs_for_subsystem(
        classified_logs_path,
        asset_id=ASSET_ID,
        system=SYSTEM,
        subsystem=SUBSYSTEM,
        date=END_DATE,
        days_back=DAYS_BACK,
        log_column=LOG_COLUMN,
    )
    print(f"{len(logs_df)} log entries found")
    display(logs_df)